In [2]:
# 1. Systeem dependencies
!apt-get update && apt-get install -y zstd

# 2. Ollama installeren
!curl -fsSL https://ollama.com/install.sh | sh

# 3. CrewAI installeren (we negeren de errors van de Google-pakketten)
!pip install -q --no-warn-conflicts crewai langchain_community
!pip install -q crewai_tools langchain_openai pypdf

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cli.github.com/packages stable/main amd64 Packages [357 B]       
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease   
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,930 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Get:13 https://ppa.launchpadcontent.ne

In [3]:
import crewai
import langchain_community
print("CrewAI is succesvol geladen!")

CrewAI is succesvol geladen!


In [7]:
from google.colab import files
uploaded = files.upload() # Selecteer hier je Marstek PDF

KeyboardInterrupt: 

In [4]:
import os
import subprocess
import time

# 1. Installeer zstd en Ollama (met forcering van het pad)
print("Bezig met installeren van dependencies...")
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Definieer het volledige pad naar ollama
# De installer zet hem meestal in /usr/local/bin/ollama
OLLAMA_PATH = "/usr/local/bin/ollama"

# 3. Start de server op de achtergrond
print("Ollama server opstarten...")
subprocess.Popen([OLLAMA_PATH, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(15) # Geef de server tijd

# 4. Pull het model met het volledige pad
print("Model downloaden (kan even duren)...")
subprocess.run([OLLAMA_PATH, "pull", "llama3"])

# 4b. Pull het embedding model voor de PDF en GitHub tools
print("Embedding model downloaden...")
subprocess.run([OLLAMA_PATH, "pull", "nomic-embed-text"])

# 5. Controleer of het model er staat
print("\nGeïnstalleerde modellen:")
subprocess.run([OLLAMA_PATH, "list"])

Bezig met installeren van dependencies...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease          
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dep

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 6a0746a1ec1a:   0% ▕                  ▏  19 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   1% ▕                  ▏  38 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   2% ▕                  ▏  91 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   3% ▕                  ▏ 148 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   5% ▕                  ▏ 210 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   5% ▕                  ▏ 234 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a


Geïnstalleerde modellen:
NAME             ID              SIZE      MODIFIED               
llama3:latest    365c0bd3c000    4.7 GB    Less than a second ago    


pulling manifest 
pulling 6a0746a1ec1a: 100% ▕██████████████████▏ 4.7 GB                         
pulling 4fa551d4f938: 100% ▕██████████████████▏  12 KB                         
pulling 8ab4849b038c: 100% ▕██████████████████▏  254 B                         
pulling 577073ffcc6c: 100% ▕██████████████████▏  110 B                         
pulling 3f8eb4da87fa: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


CompletedProcess(args=['/usr/local/bin/ollama', 'list'], returncode=0)

In [5]:
import os
from crewai import Agent, Task, Crew, Process
from crewai_tools import FileReadTool, GithubSearchTool, PDFSearchTool
from langchain_openai import ChatOpenAI

# --- CONFIGURATIE ---
# GitHub gegevens ophalen uit je secrets
GH_TOKEN = user_secrets.get_secret("GH_MARSTEK_TOKEN")
GH_USER = user_secrets.get_secret("JOUW_GEBRUIKERSNAAM")
GH_REPO = "Steavy/ha-marstek-local-api"
GH_EMAIL = user_secrets.get_secret("JOUW_EMAIL")

# Omgevingsvariabelen instellen voor Tools & Git
os.environ["OPENAI_API_KEY"] = "sk-ollama" # Fake key voor CrewAI validatie
os.environ["GITHUB_TOKEN"] = GH_TOKEN

# Lokale LLM configuratie (Ollama)
# Zorg dat 'llama3' (of jouw model) gedownload is via: ollama pull llama3
local_llm = ChatOpenAI(
    model="llama3",
    base_url="http://localhost:11434/v1",
    api_key="sk-ollama"
)

# RAG Configuratie voor Tools (gebruikt Ollama voor embeddings)
# Dit voorkomt dat de tools proberen OpenAI aan te roepen voor de PDF/GitHub scan
rag_config = {
    "embedder": {
        "provider": "ollama",
        "config": {
            "model": "nomic-embed-text", # Zorg voor: ollama pull nomic-embed-text
            "url": "http://localhost:11434/api/embeddings"
        }
    }
}

# --- TOOLS INITIALISATIE ---
pdf_tool = PDFSearchTool(
    pdf='MarstekDeviceOpenApi.pdf',
    config=rag_config
)

github_tool = GithubSearchTool(
    github_repo=f"https://github.com/{GH_REPO}",
    gh_token=GH_TOKEN,
    content_types=['code', 'issue'],
    config=rag_config
)

file_tool = FileReadTool()

# --- AGENTS ---

api_analyst = Agent(
    role='Marstek API Specialist',
    goal=f'Analyseer de Rev 2.0 PDF en vergelijk deze met de rc7 code in {GH_REPO}.',
    backstory='Je bent een expert in Marstek UDP protocollen. Je focust op byte-offsets en nieuwe JSON velden.',
    llm=local_llm,
    tools=[pdf_tool, github_tool],
    verbose=True
)

developer = Agent(
    role='Home Assistant Developer',
    goal='Update de Python integratie (sensor.py, coordinator.py) naar de nieuwe API standaard.',
    backstory='Je bent een senior Python dev die feilloos asynchrone UDP-communicatie in HA schrijft.',
    llm=local_llm,
    tools=[file_tool],
    verbose=True
)

qa_specialist = Agent(
    role='HA Configuration Specialist',
    goal='Update de UI configuratie (strings.json, services.yaml) voor de nieuwe features.',
    backstory='Je zorgt dat de vertalingen en entiteitsnamen perfect aansluiten bij de HA-standaarden.',
    llm=local_llm,
    tools=[file_tool],
    verbose=True
)

github_manager = Agent(
    role='GitHub Operations Manager',
    goal=f'Beheer de PR workflow voor {GH_REPO}.',
    backstory=f'Je zorgt voor schone commits onder de naam {GH_USER} en opent de finale PR.',
    llm=local_llm,
    tools=[github_tool],
    verbose=True
)

# --- TASKS ---

task_scan = Task(
    description='''Scan de PDF voor: 
    1. Passive mode (power, cd_time) 
    2. DOD.SET parameters 
    3. Nieuwe ES/EM velden. 
    Vergelijk dit met de huidige rc7 implementatie in de GitHub repo.''',
    agent=api_analyst,
    expected_output='Een gedetailleerd rapport van ontbrekende of gewijzigde parameters t.o.v. de rc7 versie.'
)

task_code = Task(
    description='Pas coordinator.py en sensor.py aan. Implementeer de nieuwe UDP pakketstructuur voor Passive mode.',
    agent=developer,
    context=[task_scan],
    expected_output='Bijgewerkte Python broncode klaar voor integratie.'
)

task_config = Task(
    description='Update de strings.json en services.yaml. Voeg DOD instellingen toe als nieuwe service.',
    agent=qa_specialist,
    context=[task_code],
    expected_output='Gereviseerde JSON en YAML configuratiebestanden.'
)

task_github_pr = Task(
    description=f'''Maak een branch 'upgrade-rc7-to-api-v2' aan in {GH_REPO}. 
    Commit de wijzigingen en open een Pull Request naar de master/rc7 branch.''',
    agent=github_manager,
    context=[task_config],
    human_input=True,
    expected_output='Link naar de geopende Pull Request.'
)

# --- CREW KICKOFF ---

marstek_upgrade_crew = Crew(
    agents=[api_analyst, developer, qa_specialist, github_manager],
    tasks=[task_scan, task_code, task_config, task_github_pr],
    process=Process.sequential,
    verbose=True
)

result = marstek_upgrade_crew.kickoff()
print("######################")
print("## RESULTAAT UPGRADE ##")
print("######################")
print(result)

ModuleNotFoundError: No module named 'crewai_tools'